# EfficientNet 안전고리 분류

- 설명: EfficientNet 기반 안전고리 connected/unconnected 분류 실험입니다.
- 작성자: 조하민

> 공개 저장소에는 데이터, 모델 가중치, API 키와 노트북 실행 결과를 포함하지 않습니다.


In [ ]:
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/hook_cls_raw")

folders = {
    "connected": ROOT / "connected",
    "unconnected": ROOT / "unconnected"
}

image_exts = [".jpg", ".jpeg", ".png", ".webp"]

for label, folder in folders.items():
    image_paths = sorted([
        p for p in folder.iterdir()
        if p.suffix.lower() in image_exts
    ])

    print(f"{label}: {len(image_paths)}개 파일 발견")

    # 1단계: 충돌 방지를 위해 임시 이름으로 먼저 변경
    temp_paths = []
    for i, img_path in enumerate(image_paths, start=1):
        temp_path = folder / f"temp_{label}_{i:04d}{img_path.suffix.lower()}"
        img_path.rename(temp_path)
        temp_paths.append(temp_path)

    # 2단계: 최종 이름으로 변경
    for i, temp_path in enumerate(temp_paths, start=1):
        new_name = f"{label}_{i:04d}{temp_path.suffix.lower()}"
        new_path = folder / new_name
        temp_path.rename(new_path)

print("파일명 변경 완료")

In [ ]:
# ============================================================
# 안전고리 crop 이미지 기반 체결/미체결 분류 파일럿 실험
# ------------------------------------------------------------
# 이 실험은 RF-DETR과 직접 연결하는 최종 단계가 아닙니다.
# 목적은 "안전고리 crop 이미지 1장"만 보고 connected / unconnected
# 분류가 가능한지 확인하는 파일럿 실험입니다.
#
# 데이터 불균형이 크기 때문에 accuracy만 보면 안 됩니다.
# 특히 unconnected recall, unconnected f1-score, confusion matrix를
# 중요하게 확인해야 합니다.
# ============================================================

!pip install -q scikit-learn pandas

import os
import random
import shutil
from pathlib import Path

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import datasets, transforms, models

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score
)

from google.colab import drive

In [ ]:
# 셀 2.
# ============================================================
# Google Drive 마운트
# ------------------------------------------------------------
# 원본 데이터는 Drive에 두고, split 데이터는 Colab 로컬에 복사해서 사용합니다.
# 이렇게 하면 Drive I/O 병목을 줄이면서도 best model과 CSV 결과는 보존할 수 있습니다.
# ============================================================

# Drive 안 데이터 위치 예시:
# /content/drive/MyDrive/hook_cls_raw
# ├── connected
# └── unconnected

RAW_DIR = Path("/content/drive/MyDrive/hook_cls_raw")

# 학습 중에는 로컬 /content를 쓰는 편이 일반적으로 더 빠릅니다.
SPLIT_DIR = Path("/content/hook_cls_split")

# 결과물은 Drive에 저장해서 Colab 런타임이 꺼져도 보존합니다.
OUTPUT_DIR = Path("/content/drive/MyDrive/hook_cls_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BEST_MODEL_PATH = str(OUTPUT_DIR / "best_effnet_hook_classifier.pth")
MISCLASSIFIED_CSV_PATH = str(OUTPUT_DIR / "misclassified_results.csv")

CLASS_NAMES = ["connected", "unconnected"]

# torchvision ImageFolder는 폴더명 알파벳순으로 class index를 부여합니다.
# connected -> 0, unconnected -> 1 이 되도록 폴더명을 유지합니다.

SEED = 42
IMG_SIZE = 224
BATCH_SIZE = 32
NUM_EPOCHS = 15

LR = 1e-4
WEIGHT_DECAY = 1e-4

USE_CLASS_WEIGHT_LOSS = False  # WeightedRandomSampler를 쓰므로 기본은 False
NUM_WORKERS = 2

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

set_seed(SEED)

In [ ]:
# 3. 데이터 train / valid / test 분리

# ============================================================
# 중요:
# 1. 먼저 원본 이미지를 train / valid / test로 분리합니다.
# 2. augmentation은 split 이후 train dataset transform에서만 적용합니다.
# 3. valid/test에는 절대 증강 이미지를 저장하거나 넣지 않습니다.
# ============================================================

def list_images(class_dir):
    exts = [".jpg", ".jpeg", ".png", ".bmp", ".webp"]
    return [
        p for p in class_dir.iterdir()
        if p.is_file() and p.suffix.lower() in exts
    ]

def print_raw_counts(raw_dir):
    print("Original data counts")
    for cls in CLASS_NAMES:
        cls_dir = raw_dir / cls
        count = len(list_images(cls_dir)) if cls_dir.exists() else 0
        print(f"  {cls}: {count}")

def split_class_images(image_paths, seed=42):
    """
    70 / 15 / 15 split.
    먼저 train 70%, temp 30%로 나누고,
    temp를 valid/test 15%/15%로 다시 나눕니다.
    """
    train_paths, temp_paths = train_test_split(
        image_paths,
        test_size=0.30,
        random_state=seed,
        shuffle=True
    )

    valid_paths, test_paths = train_test_split(
        temp_paths,
        test_size=0.50,
        random_state=seed,
        shuffle=True
    )

    return train_paths, valid_paths, test_paths

def copy_images(paths, dst_dir):
    dst_dir.mkdir(parents=True, exist_ok=True)

    for src_path in paths:
        dst_path = dst_dir / src_path.name

        # 같은 이름 파일 충돌 방지
        if dst_path.exists():
            stem = src_path.stem
            suffix = src_path.suffix
            i = 1
            while True:
                new_dst_path = dst_dir / f"{stem}_{i}{suffix}"
                if not new_dst_path.exists():
                    dst_path = new_dst_path
                    break
                i += 1

        shutil.copy2(src_path, dst_path)

def print_split_counts(split_dir):
    for split in ["train", "valid", "test"]:
        print(f"\n[{split}]")
        for cls in CLASS_NAMES:
            cls_dir = split_dir / split / cls
            count = len(list_images(cls_dir))
            print(f"  {cls}: {count}")

def make_split_dataset(raw_dir, split_dir, seed=42, overwrite=True):
    if overwrite and split_dir.exists():
        shutil.rmtree(split_dir)

    for split in ["train", "valid", "test"]:
        for cls in CLASS_NAMES:
            (split_dir / split / cls).mkdir(parents=True, exist_ok=True)

    print_raw_counts(raw_dir)

    for cls in CLASS_NAMES:
        cls_dir = raw_dir / cls
        image_paths = list_images(cls_dir)

        if len(image_paths) == 0:
            raise ValueError(f"No images found in: {cls_dir}")

        train_paths, valid_paths, test_paths = split_class_images(
            image_paths,
            seed=seed
        )

        copy_images(train_paths, split_dir / "train" / cls)
        copy_images(valid_paths, split_dir / "valid" / cls)
        copy_images(test_paths, split_dir / "test" / cls)

    print("\nSplit data counts")
    print_split_counts(split_dir)

make_split_dataset(
    raw_dir=RAW_DIR,
    split_dir=SPLIT_DIR,
    seed=SEED,
    overwrite=True
)

In [ ]:
# 셀 4. transform 정의

# ============================================================
# train augmentation
# ------------------------------------------------------------
# RandomCrop, CutMix, MixUp, Cutout, RandomErasing, 강한 가림 증강은
# 사용하지 않습니다.
#
# 이유:
# 안전고리 체결 여부를 판단하는 연결부가 잘리거나 가려지면
# 라벨 의미 자체가 깨질 수 있기 때문입니다.
# ============================================================

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomAffine(
        degrees=10,
        translate=(0.05, 0.05),
        scale=(0.95, 1.05)
    ),
    transforms.ColorJitter(
        brightness=0.15,
        contrast=0.15,
        saturation=0.10,
        hue=0.03
    ),
    transforms.GaussianBlur(
        kernel_size=3,
        sigma=(0.1, 0.8)
    ),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

valid_test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

In [ ]:
# 셀 5. dataset / dataloader 생성

train_dataset = datasets.ImageFolder(
    root=SPLIT_DIR / "train",
    transform=train_transform
)

valid_dataset = datasets.ImageFolder(
    root=SPLIT_DIR / "valid",
    transform=valid_test_transform
)

test_dataset = datasets.ImageFolder(
    root=SPLIT_DIR / "test",
    transform=valid_test_transform
)

print("Class to index:", train_dataset.class_to_idx)
print("Index to class:", {v: k for k, v in train_dataset.class_to_idx.items()})

connected_idx = train_dataset.class_to_idx["connected"]
unconnected_idx = train_dataset.class_to_idx["unconnected"]

# =========================
# WeightedRandomSampler
# =========================
# 데이터 불균형이 크므로 train loader에서 unconnected가 더 자주 샘플링되도록 합니다.

train_targets = np.array(train_dataset.targets)
class_counts = np.bincount(train_targets)

print("\nTrain class counts")
for idx, count in enumerate(class_counts):
    print(f"  {train_dataset.classes[idx]}: {count}")

class_sample_weights = 1.0 / class_counts
sample_weights = class_sample_weights[train_targets]
sample_weights = torch.DoubleTensor(sample_weights)

sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    sampler=sampler,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

In [ ]:
# 셀 6. EfficientNet-B0 모델 정의

def build_model(num_classes=2):
    weights = models.EfficientNet_B0_Weights.IMAGENET1K_V1
    model = models.efficientnet_b0(weights=weights)

    in_features = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(in_features, num_classes)

    return model

model = build_model(num_classes=2).to(device)

# =========================
# Loss 설정
# =========================
# WeightedRandomSampler를 사용하므로 기본은 class weight loss를 끕니다.
# 필요하면 USE_CLASS_WEIGHT_LOSS=True로 바꿔 실험할 수 있습니다.

if USE_CLASS_WEIGHT_LOSS:
    train_targets = np.array(train_dataset.targets)
    class_counts = np.bincount(train_targets)
    class_weights = 1.0 / class_counts
    class_weights = class_weights / class_weights.sum() * len(class_counts)
    class_weights = torch.tensor(class_weights, dtype=torch.float32).to(device)

    criterion = nn.CrossEntropyLoss(weight=class_weights)
    print("Using class weighted CrossEntropyLoss:", class_weights)
else:
    criterion = nn.CrossEntropyLoss()
    print("Using normal CrossEntropyLoss")

optimizer = optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY
)

In [ ]:
# 셀 7. train / evaluate 함수 정의

def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()

    running_loss = 0.0
    all_preds = []
    all_labels = []

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

        preds = torch.argmax(outputs, dim=1)

        all_preds.extend(preds.detach().cpu().numpy())
        all_labels.extend(labels.detach().cpu().numpy())

    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)

    return epoch_loss, epoch_acc

@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()

    running_loss = 0.0
    all_preds = []
    all_labels = []
    all_probs = []

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        probs = torch.softmax(outputs, dim=1)
        preds = torch.argmax(probs, dim=1)

        running_loss += loss.item() * images.size(0)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)

    return {
        "loss": epoch_loss,
        "accuracy": epoch_acc,
        "labels": np.array(all_labels),
        "preds": np.array(all_preds),
        "probs": np.array(all_probs)
    }

def get_unconnected_f1(y_true, y_pred):
    report_dict = classification_report(
        y_true,
        y_pred,
        target_names=CLASS_NAMES,
        output_dict=True,
        zero_division=0
    )
    return report_dict["unconnected"]["f1-score"]

In [ ]:
# 셀 8. 학습 실행 및 best model 저장

best_unconnected_f1 = -1.0

for epoch in range(1, NUM_EPOCHS + 1):
    train_loss, train_acc = train_one_epoch(
        model=model,
        loader=train_loader,
        criterion=criterion,
        optimizer=optimizer,
        device=device
    )

    valid_result = evaluate(
        model=model,
        loader=valid_loader,
        criterion=criterion,
        device=device
    )

    valid_loss = valid_result["loss"]
    valid_acc = valid_result["accuracy"]
    valid_labels = valid_result["labels"]
    valid_preds = valid_result["preds"]

    valid_unconnected_f1 = get_unconnected_f1(valid_labels, valid_preds)

    print("=" * 70)
    print(f"Epoch [{epoch}/{NUM_EPOCHS}]")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Valid Loss: {valid_loss:.4f} | Valid Acc: {valid_acc:.4f}")
    print(f"Valid unconnected F1-score: {valid_unconnected_f1:.4f}")

    print("\nValidation classification report")
    print(classification_report(
        valid_labels,
        valid_preds,
        target_names=CLASS_NAMES,
        zero_division=0
    ))

    # best model은 accuracy가 아니라 valid unconnected f1-score 기준으로 저장합니다.
    if valid_unconnected_f1 > best_unconnected_f1:
        best_unconnected_f1 = valid_unconnected_f1

        torch.save({
            "model_state_dict": model.state_dict(),
            "class_to_idx": train_dataset.class_to_idx,
            "epoch": epoch,
            "best_unconnected_f1": best_unconnected_f1
        }, BEST_MODEL_PATH)

        print(f"Best model saved: {BEST_MODEL_PATH}")
        print(f"Best valid unconnected F1-score: {best_unconnected_f1:.4f}")

In [ ]:
# 셀 9. test set 최종 평가

# ============================================================
# Test 평가
# ------------------------------------------------------------
# test accuracy도 출력하지만,
# 안전 문제에서는 accuracy보다 unconnected recall과 unconnected f1-score,
# 그리고 confusion matrix를 더 중요하게 봐야 합니다.
#
# 특히 "실제 unconnected인데 connected로 예측하는 경우"가 위험합니다.
# ============================================================

checkpoint = torch.load(BEST_MODEL_PATH, map_location=device)

best_model = build_model(num_classes=2).to(device)
best_model.load_state_dict(checkpoint["model_state_dict"])
best_model.eval()

test_result = evaluate(
    model=best_model,
    loader=test_loader,
    criterion=criterion,
    device=device
)

test_labels = test_result["labels"]
test_preds = test_result["preds"]
test_probs = test_result["probs"]
test_acc = test_result["accuracy"]

print("=" * 70)
print("Final Test Evaluation")
print(f"Test Accuracy: {test_acc:.4f}")

print("\nConfusion Matrix")
print(confusion_matrix(test_labels, test_preds))

print("\nClassification Report")
print(classification_report(
    test_labels,
    test_preds,
    target_names=CLASS_NAMES,
    zero_division=0
))

In [ ]:
# 셀 10. threshold별 평가
# ============================================================
# Threshold 실험
# ------------------------------------------------------------
# 기본 argmax 대신 unconnected 확률 기준 threshold를 조정합니다.
#
# unconnected_prob >= threshold 이면 unconnected로 예측
# 아니면 connected로 예측합니다.
#
# 안전 문제에서는 unconnected recall을 높이는 것이 중요합니다.
# threshold를 낮추면 unconnected로 더 쉽게 판단하므로
# 미체결 recall이 올라갈 수 있습니다.
# 단, connected를 unconnected로 잘못 예측하는 false positive는 늘 수 있습니다.
# ============================================================

def evaluate_with_unconnected_threshold(
    y_true,
    probs,
    threshold,
    class_names=CLASS_NAMES
):
    connected_idx = class_names.index("connected")
    unconnected_idx = class_names.index("unconnected")

    unconnected_probs = probs[:, unconnected_idx]

    threshold_preds = np.where(
        unconnected_probs >= threshold,
        unconnected_idx,
        connected_idx
    )

    acc = accuracy_score(y_true, threshold_preds)

    print("=" * 70)
    print(f"Threshold evaluation | unconnected threshold = {threshold}")
    print(f"Accuracy: {acc:.4f}")

    print("\nConfusion Matrix")
    print(confusion_matrix(y_true, threshold_preds))

    print("\nClassification Report")
    print(classification_report(
        y_true,
        threshold_preds,
        target_names=class_names,
        zero_division=0
    ))

    return threshold_preds

for th in [0.3, 0.4, 0.5, 0.6]:
    _ = evaluate_with_unconnected_threshold(
        y_true=test_labels,
        probs=test_probs,
        threshold=th,
        class_names=CLASS_NAMES
    )

In [ ]:
# 셀 11. misclassified_results.csv 저장

# ============================================================
# 오분류 test 이미지 목록 저장
# ------------------------------------------------------------
# CSV 컬럼:
# image_path, true_label, pred_label, connected_prob, unconnected_prob
# ============================================================

def save_misclassified_csv(
    dataset,
    y_true,
    y_pred,
    probs,
    csv_path,
    class_names=CLASS_NAMES
):
    rows = []

    # ImageFolder dataset.samples:
    # [(image_path, class_idx), ...]
    image_paths = [sample[0] for sample in dataset.samples]

    connected_idx = class_names.index("connected")
    unconnected_idx = class_names.index("unconnected")

    for image_path, true_idx, pred_idx, prob in zip(
        image_paths,
        y_true,
        y_pred,
        probs
    ):
        if true_idx != pred_idx:
            rows.append({
                "image_path": image_path,
                "true_label": class_names[true_idx],
                "pred_label": class_names[pred_idx],
                "connected_prob": float(prob[connected_idx]),
                "unconnected_prob": float(prob[unconnected_idx])
            })

    df = pd.DataFrame(rows, columns=[
        "image_path",
        "true_label",
        "pred_label",
        "connected_prob",
        "unconnected_prob"
    ])

    df.to_csv(csv_path, index=False, encoding="utf-8-sig")

    print(f"Misclassified results saved to: {csv_path}")
    print(f"Number of misclassified images: {len(df)}")

    return df

misclassified_df = save_misclassified_csv(
    dataset=test_dataset,
    y_true=test_labels,
    y_pred=test_preds,
    probs=test_probs,
    csv_path=MISCLASSIFIED_CSV_PATH,
    class_names=CLASS_NAMES
)

misclassified_df.head()

In [ ]:
!unzip -q /content/분류_new_test.zip -d /content/분류_new_test

In [ ]:
# ============================================================
# 셀 13. 새로운 이미지 1장 예측
# ============================================================

from PIL import Image
import torch
import numpy as np

# 새로 테스트할 이미지 경로
# Drive에 테스트 이미지를 올려두고 경로만 바꿔주면 됨
NEW_IMAGE_PATH = "/content/분류_new_test/new_test/aihub체결테스트2.jpg"

def predict_new_image(
    image_path,
    model,
    transform,
    device,
    threshold=0.4,
    class_names=CLASS_NAMES
):
    model.eval()

    # 이미지 열기
    image = Image.open(image_path).convert("RGB")

    # 학습 때와 같은 전처리 적용
    x = transform(image).unsqueeze(0).to(device)

    with torch.no_grad():
        outputs = model(x)
        probs = torch.softmax(outputs, dim=1)[0].cpu().numpy()

    connected_idx = class_names.index("connected")
    unconnected_idx = class_names.index("unconnected")

    connected_prob = float(probs[connected_idx])
    unconnected_prob = float(probs[unconnected_idx])

    # threshold 기준 예측
    if unconnected_prob >= threshold:
        pred_label = "unconnected"
    else:
        pred_label = "connected"

    print("=" * 70)
    print("New Image Prediction")
    print("image_path:", image_path)
    print(f"connected_prob: {connected_prob:.4f}")
    print(f"unconnected_prob: {unconnected_prob:.4f}")
    print(f"threshold: {threshold}")
    print("prediction:", pred_label)

    return {
        "image_path": image_path,
        "pred_label": pred_label,
        "connected_prob": connected_prob,
        "unconnected_prob": unconnected_prob,
        "threshold": threshold
    }

result = predict_new_image(
    image_path=NEW_IMAGE_PATH,
    model=best_model,
    transform=valid_test_transform,
    device=device,
    threshold=0.4
)

result